> **Solutions.** This is the tutorial notebook with every exercise filled in.
> Try them yourself in `from_urdf_to_semantic_world.ipynb` first.

# From URDF to a Semantic Digital Twin

*IJCAI 2026 workshop — hands-on tutorial (150 minutes)*

A URDF tells a robot **where** every link is. It does not tell the robot **what** any of
them is. `<link name="cabinet6_drawer_top"/>` is a name a human chose; to the robot it is a
rigid body on a prismatic joint, indistinguishable from a sliding door or a telescopic mast.

So "open the drawer" is not a question a URDF can answer.

This tutorial closes that gap using the
[Semantic Digital Twin](https://github.com/cram2/cognitive_robot_abstract_machine), a world
model that carries geometry, kinematics **and** meaning in one structure.

| § | What we do |
|---|---|
| 1 | Load an apartment URDF and watch two reasonable heuristics both get it wrong |
| 2 | Say what things *are*: build a dresser from typed semantic annotations |
| 3 | Let the `WorldReasoner` find the drawers itself — and explain why |
| 4 | Ask the world questions with the Entity Query Language |
| 5 | Bring a robot into the world, query its typed parts, and move it |
| 6 | Ask spatial questions that span the robot and the furniture |
| 7 | Find what the reasoner is still missing, and why |

**Prerequisites:** Python, and having seen a URDF before. No prior knowledge of CRAM,
ontologies, or rule-based reasoning is assumed.

## 0. Setup

> **Check your kernel.** The top right of this notebook must say **CRAM**. If it says
> anything else, use *Kernel → Change Kernel… → CRAM*. The default Python kernel does not
> have the semantic digital twin installed.

Run the cell below. It should print a version and two `OK` lines.

In [ ]:
import logging
from collections import Counter
from pathlib import Path
from importlib.resources import files

import numpy as np

import semantic_digital_twin
from semantic_digital_twin.adapters.package_resolver import CompositePathResolver
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.api import BodySpecification, RevoluteConnectionSpecification, RobotSpecification
from semantic_digital_twin.exceptions import ExerciseVerificationFailed
from semantic_digital_twin.reasoning.predicates import is_supported_by, reachable
from semantic_digital_twin.reasoning import world_rdr
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner
from semantic_digital_twin.robots.hsrb import HSRB, HSRBArm, HSRBJoint
from semantic_digital_twin.robots.minimal_robot import MinimalRobot
from semantic_digital_twin.robots.pr2 import PR2, PR2Joint, PR2LeftArm, PR2RightArm, PR2Torso
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Door,
    Drawer,
    Dresser,
    Handle,
    Slider,
    Wardrobe,
)
from semantic_digital_twin.spatial_computations.raytracer import RayTracer
from semantic_digital_twin.spatial_types.spatial_types import (
    HomogeneousTransformationMatrix,
    Vector3,
)
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.connections import PrismaticConnection
from semantic_digital_twin.world_description.geometry import Scale

from krrood.entity_query_language.explanation.explanation import explain_inference
from krrood.entity_query_language.factories import an, contains, entity, the, variable
from krrood.entity_query_language.verbalization.pipeline import verbalize_expression

logging.disable(logging.CRITICAL)  # keep the notebook output readable

print("semantic_digital_twin", semantic_digital_twin.__version__)

URDF_DIR = Path(files("semantic_digital_twin")).parent.parent / "resources" / "urdf"
APARTMENT = URDF_DIR / "apartment.urdf"
print("OK  apartment URDF:", APARTMENT.name)

# The apartment references its meshes as package://iai_apartment/... , which comes from the
# iai_maps ROS package. If this line fails, the ROS workspace was not sourced.
CompositePathResolver().resolve("package://iai_apartment/meshes/visual/walls.dae")
print("OK  mesh packages resolve")

## 1. What a URDF can and cannot tell a robot

`URDFParser` reads a URDF into a `World`. A `World` is a graph: bodies and regions as
nodes, connections (joints) as edges, with a registry of degrees of freedom.

*(This URDF puts a `material` tag inside a `collision` element, which is not legal there, so
the XML parser prints an `Unknown tag` warning. It is harmless — real-world URDFs are rarely
clean.)*

In [ ]:
world = URDFParser.from_file(str(APARTMENT)).parse()

print("bodies     ", len(world.bodies))
print("connections", len(world.connections))
print(Counter(type(c).__name__ for c in world.connections))

Let's look at it. `RayTracer` renders the world straight into the notebook — no simulator,
no ROS, no window manager.

*(Drag to orbit, scroll to zoom.)*

In [ ]:
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Now the important part

Ask the world what it *means*:

In [ ]:
print(world.semantic_annotations)

Empty. A world parsed from a file is a purely kinematic model. Nothing in it is a drawer, a
handle, or a door — those are things *we* read into the link names.

### Exercise 1 — find everything the robot can open

You are writing the perception layer for a robot in this apartment. It needs a list of
drawers. There are two obvious ways to guess, and they are both reasonable.

**Guess A — match the names.** Somebody called them drawers, so search for that:

In [ ]:
by_name = {body.name.name for body in world.bodies if "drawer" in body.name.name.lower()}
print(len(by_name), "bodies matched by name")

**Guess B — match the structure.** A drawer slides, so look for prismatic joints:

In [ ]:
sliding = {c.child.name.name for c in world.connections if isinstance(c, PrismaticConnection)}
print(len(sliding), "bodies sit on a prismatic joint")

Both say 25. That looks like confirmation — two independent methods agreeing.

It isn't. Compare the actual sets:

In [ ]:
print("both agree on   :", len(by_name & sliding))
print()
print("named, not sliding:", sorted(by_name - sliding))
print("sliding, not named:", sorted(sliding - by_name))

The two heuristics agree on a *number* and disagree about *which bodies*, and each is wrong
in its own way:

- `handle_cab1_drawer_bottom` and `handle_cab1_drawer_mid` are **handles**. They matched
  only because the word "drawer" appears in the name of the drawer they belong to.
- `cabinet2_door_out_fancy` and `cabinet3_door_bottom_out_fancy` are **doors** that slide
  out before they swing open. They are on prismatic joints, but you cannot open them like a
  drawer.

Neither 25 is the right answer. Hold on to that number — we will find out what the truth is
in §7, and it is neither of these.

And the name-based guess has a deeper problem than being wrong: it only works at all because
a human happened to name these links in English. Rename them `link_001 … link_113` — a
perfectly legal URDF, and what most CAD exporters give you — and it returns nothing.

The rest of this tutorial is about getting an answer that does not depend on either luck.

## 2. Saying what things are

A **semantic annotation** attaches meaning to bodies in a world: *this* body is a handle,
*these* bodies together are a drawer.

The library's position is worth stating plainly, because it is a design choice you may want
to argue with. Annotations are inspired by ontologies, but they are **not** an ontology:
there is no OWL, no RDF, no triple store, no separate reasoner. An annotation is an ordinary
Python dataclass, and reasoning is done with Python's type system plus a query language. The
claim is that you get the expressiveness without the impedance mismatch; the cost is that
your knowledge lives in Python rather than in a portable standard.

Here is a real one from the library:

```python
@dataclass(eq=False)
class Drawer(Furniture, HasCaseAsRootBody, HasHandle, HasMechanicalJoint):
    @classproperty
    def hole_direction(self) -> Vector3:
        return Vector3.Z()
```

The mixins *are* the definition: a drawer is furniture, it has a case as its root body, it
has a handle, and it has a mechanical joint. Remember that definition — in §7 it turns out
to be slightly too strict.

Building one of these takes two ingredients you haven't used yet: a way to place bodies in
space, and the `add` method that wires annotations together. Transforms first.

### A quick word on transforms

Every `world_root_T_self=...` you will write below is a `HomogeneousTransformationMatrix` — a
4×4 matrix saying where one frame sits relative to another. The library's convention, used
everywhere: read `A_T_B` as *"the pose of B, expressed in frame A."* (The full story is in
`examples/using_transformations.md`, linked again at the end of this notebook.)

Two things about them are enough for what follows:

- `HomogeneousTransformationMatrix.from_xyz_rpy(x=.., y=.., z=.., roll=.., pitch=.., yaw=..)`
  builds one from a position and Euler angles — everything defaults to `0`.
- Transforms compose with `@`. If you know `world_T_a` and `a_T_b`, then `world_T_a @ a_T_b`
  is `world_T_b` — the pose of `b`, expressed in `world`.

In [ ]:
world_T_a = HomogeneousTransformationMatrix.from_xyz_rpy(x=1.0)
a_T_b = HomogeneousTransformationMatrix.from_xyz_rpy(y=0.5)
world_T_b = world_T_a @ a_T_b

print("world_T_b position:", world_T_b.to_position().to_np().flatten()[:3])

### Exercise 2 — build a transform

Below you will place a handle 0.28 m in front of a drawer's origin (negative `x`). Practice
that transform first.

Build a `HomogeneousTransformationMatrix` with `from_xyz_rpy` that has no rotation and sits
0.28 m in the negative `x` direction. Assign it to `handle_offset`.

In [ ]:
handle_offset = HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.28)

In [ ]:
# Run this to check your answer.
if handle_offset is ... or not isinstance(handle_offset, HomogeneousTransformationMatrix):
    raise ExerciseVerificationFailed(
        "handle_offset should be a HomogeneousTransformationMatrix."
    )

position = handle_offset.to_position().to_np().flatten()[:3]
expected = np.array([-0.28, 0.0, 0.0])
if not np.allclose(position, expected, atol=1e-6):
    raise ExerciseVerificationFailed(f"Expected position {expected}, got {position}.")

print("Correct.")

### Building a drawer

`create_with_new_body_in_world` spawns the annotation, a body, and its geometry in one call.
Then `add` wires the parts together — one method, routed by type.

In [ ]:
drawer_world = World.create_with_root_body()

with drawer_world.modify_world():
    drawer = Drawer.create_with_new_body_in_world(
        name="drawer",
        scale=Scale(0.5, 0.5, 0.4),
        world=drawer_world,
        world_root_T_self=HomogeneousTransformationMatrix(),
    )
    handle = Handle.create_with_new_body_in_world(
        name="drawer_handle",
        scale=Scale(0.05, 0.25, 0.1),
        world_root_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.25),
        world=drawer_world,
    )
    slider = Slider.create_with_new_body_in_world(
        name="drawer_slider",
        world_root_T_self=HomogeneousTransformationMatrix(),
        world=drawer_world,
        parent_connection_specification=Slider.parent_connection_specification(
            axis=Vector3.NEGATIVE_X()
        ),
    )

    # One method. It matches each part against the typed part-whole fields of the whole.
    drawer.add(handle)   # -> drawer.handle           (single-valued field)
    drawer.add(slider)   # -> drawer.mechanical_joint (single-valued field)

print("drawer.handle           is handle:", drawer.handle is handle)
print("drawer.mechanical_joint is slider:", drawer.mechanical_joint is slider)

Note what `add` did *not* need: no `parent=`, no `child=`, no joint declaration. The part's
type was enough to decide both which field it belongs in and where it mounts in the
kinematic tree. You can see the tree it built:

In [ ]:
drawer_world.visualize_world_structure()

`visualize_world_structure` is the most useful debugging tool in the library — it draws the
kinematic tree as the robot actually sees it. (Try it on `world`, the apartment, if you
like: 113 bodies make a very wide image.)

### The payoff

The annotation is not a label sitting beside the geometry — it is wired into it. `Drawer`
has a `mechanical_joint`, so a drawer can be opened. Here is what you built:

In [ ]:
ray_tracer = RayTracer(drawer_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

`add` made the handle a kinematic child of the drawer, so it will always move with the
drawer's front. There is nothing else in this scene for the drawer to slide out of yet, so
you cannot see that for real — but you are about to build a case for it to open out of, which
is exactly what "open the drawer" needs: one line of code, `mechanical_joint.position = ...`,
and the geometry follows.

### Exercise 3 — build a dresser

Build a `Dresser` containing a drawer like the one you just built by hand. You need a
`Dresser`, a `Drawer`, a `Handle` and a `Slider`, wired together with `add` exactly as above.

One thing is new: a `Dresser` is a case, and a case only opens on one side —
`Dresser.hole_direction` says which. Get the slider's axis to match, or the drawer will slide
into the back of the case instead of out through the front.

Then open the drawer and render.

In [ ]:
dresser_world = World.create_with_root_body()

with dresser_world.modify_world():
    dresser = Dresser.create_with_new_body_in_world(
        name="dresser",
        scale=Scale(0.6, 0.6, 0.5),
        world=dresser_world,
        world_root_T_self=HomogeneousTransformationMatrix(),
    )
    dresser_drawer = Drawer.create_with_new_body_in_world(
        name="dresser_drawer",
        scale=Scale(0.5, 0.5, 0.4),
        world=dresser_world,
        world_root_T_self=HomogeneousTransformationMatrix(),
    )
    dresser_handle = Handle.create_with_new_body_in_world(
        name="dresser_drawer_handle",
        scale=Scale(0.05, 0.25, 0.3),
        world_root_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.28),
        world=dresser_world,
    )
    dresser_slider = Slider.create_with_new_body_in_world(
        name="dresser_drawer_slider",
        world_root_T_self=HomogeneousTransformationMatrix(),
        world=dresser_world,
        parent_connection_specification=Slider.parent_connection_specification(
            axis=Dresser.hole_direction
        ),
    )

    dresser_drawer.add(dresser_handle)
    dresser_drawer.add(dresser_slider)
    dresser.add(dresser_drawer)

dresser_drawer.mechanical_joint.position = 0.25

ray_tracer = RayTracer(dresser_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

In [ ]:
# Run this to check your answer.
if dresser is ... or not isinstance(dresser, Dresser):
    raise ExerciseVerificationFailed("dresser should be a Dresser.")
if dresser_drawer is ... or not isinstance(dresser_drawer, Drawer):
    raise ExerciseVerificationFailed("dresser_drawer should be a Drawer.")
if dresser_drawer.handle is not dresser_handle:
    raise ExerciseVerificationFailed(
        "Use dresser_drawer.add(dresser_handle) to attach the handle."
    )
if dresser_drawer.mechanical_joint is not dresser_slider:
    raise ExerciseVerificationFailed(
        "Use dresser_drawer.add(dresser_slider) to attach the slider."
    )
if dresser_drawer not in dresser.drawers:
    raise ExerciseVerificationFailed(
        "Use dresser.add(dresser_drawer) so the dresser owns the drawer."
    )
if dresser_drawer.mechanical_joint.position == 0:
    raise ExerciseVerificationFailed(
        "Open the drawer — set dresser_drawer.mechanical_joint.position to something nonzero."
    )

slider_axis = dresser_drawer.mechanical_joint.root.parent_connection.axis.to_np().flatten()[:3]
hole_direction = Dresser.hole_direction.to_np().flatten()[:3]
if slider_axis @ hole_direction <= 0:
    raise ExerciseVerificationFailed(
        "The slider's axis should point the same way as Dresser.hole_direction, or opening "
        "the drawer pushes it into the back of the case instead of out the front."
    )

print("Correct.")
ray_tracer = RayTracer(dresser_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### One detail that matters later

Annotations are declared `@dataclass(eq=False)`. That looks like boilerplate, but it is
load-bearing: the base class defines equality and hashing **structurally**, over the type and
the bodies referenced. Two separately constructed `Handle` objects on the same body are the
same handle:

In [ ]:
first = Handle(root=handle.root)
second = Handle(root=handle.root)

print("different objects:", id(first) != id(second))
print("but equal        :", first == second)
print("and same hash    :", hash(first) == hash(second))

Without this, the reasoner in §3 could not tell a newly inferred annotation from one it had
already found, and every run would pile up duplicates.

## 3. Not annotating by hand

Hand-annotating a dresser took twenty lines. The apartment has 113 bodies, and a building has
thousands. This does not scale, and it is not supposed to.

`WorldReasoner` applies a body of rules to a raw world and infers the annotations itself. It
runs on the apartment we loaded in §1 — the plain URDF, with nothing added.

In [ ]:
if "world" not in globals():
    world = URDFParser.from_file(str(APARTMENT)).parse()

reasoner = WorldReasoner(world)
inferred = reasoner.reason()["semantic_annotations"]

print(f"{len(inferred)} annotations inferred\n")
print(Counter(type(a).__name__ for a in inferred))

From a file that contained none of those words as *concepts*, the reasoner produced handles,
drawers, doors, and the cabinets they belong to.

So what does it say about Exercise 1?

In [ ]:
drawers = world.get_semantic_annotations_by_type(Drawer)

print(f"{len(drawers)} drawers\n")
for d in drawers:
    print("   ", d.root.name.name)

**Nineteen** — *fewer* than either heuristic's 25.

That is worth pausing on, because it looks like the reasoner did worse. It did not. The two
heuristics returned 25 bodies they could not justify; the reasoner returned 19 it can. It is
being conservative: it only claims what its rules actually support.

Which raises the obvious question, and the reason any of this is usable.

### Why does it think that is a drawer?

The reasoner is not a network. It can show its work.

In [ ]:
from krrood.entity_query_language.explanation.explanation import explain_inference
from krrood.entity_query_language.verbalization.pipeline import verbalize_expression

explanation = explain_inference(drawers[0])

print(explanation.get_satisfied_conditions_as_string())

In [ ]:
print(verbalize_expression(explanation.query_root))

Read that rule closely, because it is doing something neither heuristic in §1 could:

> If there's a FixedConnection whose parent is the child of a PrismaticConnection, there's a
> Handle whose root is the child of the FixedConnection, then there's a Drawer whose root is
> the parent of the FixedConnection, and whose handle is the Handle.

It is **structural**: "a body that slides, with a handle rigidly attached to it." Rename every
link to `link_042` and this rule still fires. That is also exactly why the two sliding doors
from §1 were excluded — they slide, but what is attached to them is not a handle.

The rules are not a black box and not a trained artifact. They are generated Python, checked
into the repository next to the code, so they are reviewed, versioned and migrated like
everything else:

In [ ]:
print(Path(world_rdr.__file__).parent)
for f in sorted(Path(world_rdr.__file__).parent.glob("*.py")):
    print("   ", f.name)

### An honest caveat

Not every rule is structural. The `Handle` rule, which the `Drawer` rule depends on, still
matches on the name — it looks for `"handle"` in the body name. In this apartment that
happens to be exactly right, all 29 of them, which is luck rather than design.

Keep both facts in mind: the drawer rule is structural, and it rests on a name-based one.

## 4. Asking the world questions

Now that the apartment carries meaning, we can query it. The **Entity Query Language** (EQL)
runs over plain Python objects — no database, no schema, no serialization step.

Three pieces: `variable` declares what you are quantifying over and from which collection,
`entity` says what you want back, and `an` / `the` evaluate it.

In [ ]:
if "world" not in globals():
    world = URDFParser.from_file(str(APARTMENT)).parse()
if not world.semantic_annotations:
    WorldReasoner(world).reason()

handle_variable = variable(Handle, world.semantic_annotations)
handles = list(an(entity(handle_variable)).evaluate())

print("handles:", len(handles))

`.where(...)` adds conditions. Conditions are written against the variable, and may walk its
attributes. `an(...)` returns every match; `the(...)` — used below — asserts there is exactly
one.

In [ ]:
drawer_variable = variable(Drawer, world.semantic_annotations)

query = an(entity(drawer_variable).where(
    contains(drawer_variable.root.name.name.lower(), "cabinet6")
))

for d in query.evaluate():
    print(d.root.name.name)

### Queries that follow part-whole structure

The reasoner also inferred which cabinet each drawer belongs to, so we can ask questions that
span several objects:

In [ ]:
for w in world.get_semantic_annotations_by_type(Wardrobe):
    print(f"{w.root.name.name:12s} {len(w.drawers)} drawers")

### Opening a real drawer

Everything from §2 applies to the inferred annotations too — they are the same classes:

In [ ]:
target = the(entity(variable(Drawer, world.semantic_annotations))).first()
slider_connection = target.root.parent_connection

print("drawer ", target.root.name.name)
print("handle ", target.handle.root.name.name)
print("joint  ", type(slider_connection).__name__)
print("range  ", slider_connection.dof.limits.lower.position,
      "->", slider_connection.dof.limits.upper.position)

before = target.handle.root.global_pose.to_position().to_np().flatten()[:3]
slider_connection.position = slider_connection.dof.limits.upper.position
after = target.handle.root.global_pose.to_position().to_np().flatten()[:3]

print("moved  ", before, "->", after)

In [ ]:
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Exercise 4

1. Write an EQL query returning every `Door` in the apartment; assign the result to `doors`.
2. Several cabinets tie for the most drawers. Find that maximum count, assign it to
   `most_drawers`, and assign the list of every wardrobe holding that many to `roomiest`.
3. Open every drawer in `roomiest` to its upper limit, and render the result.

The upper limit of a drawer's joint is
`drawer.root.parent_connection.dof.limits.upper.position`.

In [ ]:
# 1. every door
door_variable = variable(Door, world.semantic_annotations)
doors = list(an(entity(door_variable)).evaluate())
for d in doors:
    print(d.root.name.name)

# 2. the wardrobes with the most drawers
wardrobes = world.get_semantic_annotations_by_type(Wardrobe)
most_drawers = max(len(w.drawers) for w in wardrobes)
roomiest = [w for w in wardrobes if len(w.drawers) == most_drawers]
print(f"\n{len(roomiest)} wardrobes hold {most_drawers} drawers each:",
      [w.root.name.name for w in roomiest])

# 3. open all of them
for w in roomiest:
    for d in w.drawers:
        connection = d.root.parent_connection
        connection.position = connection.dof.limits.upper.position

In [ ]:
# Run this to check your answer.
if doors is ... or len(list(doors)) != 8:
    raise ExerciseVerificationFailed("There are 8 doors in this apartment.")
if not all(isinstance(d, Door) for d in doors):
    raise ExerciseVerificationFailed("doors should contain only Door annotations.")
if most_drawers != 3:
    raise ExerciseVerificationFailed("The largest number of drawers in one wardrobe is 3.")
if roomiest is ... or len(roomiest) != 5:
    raise ExerciseVerificationFailed("5 wardrobes tie for the most drawers.")
for w in roomiest:
    for d in w.drawers:
        connection = d.root.parent_connection
        if abs(connection.position - connection.dof.limits.upper.position) > 1e-6:
            raise ExerciseVerificationFailed("Every drawer in roomiest should be fully open.")

print("Correct.")
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

## 5. Robots

Everything so far has lived in a static apartment. A robot is just another URDF — parsed the
same way, and merged into the *same* world the apartment already lives in, not a separate
one.

In [ ]:
if "world" not in globals():
    world = URDFParser.from_file(str(APARTMENT)).parse()

pr2 = RobotSpecification(
    semantic_annotation_type=PR2,
    world_T_odom=HomogeneousTransformationMatrix.from_xyz_rpy(x=9.0, y=2.5),
).spawn(world)

print("bodies in world now:", len(world.bodies))

ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

*(The PR2's own URDF trips a couple of harmless parser warnings, same as the apartment's
`material`-in-`collision` tag back in §1.)*

`spawn` did two things: it parsed the PR2's URDF and merged it into `world` — the same world
you have been building up all along, drawers and all — and it annotated the robot while doing
so. `pr2` is itself a semantic annotation, and like the dresser in §2 it decomposes into typed
parts you can query the same way you queried furniture:

In [ ]:
torso = world.get_semantic_annotations_by_type(PR2Torso)[0]
left_arm = world.get_semantic_annotations_by_type(PR2LeftArm)[0]

print("pr2 is a semantic annotation of type:", type(pr2).__name__)
print("torso root        :", torso.root.name.name)
print("left arm's gripper :", left_arm.end_effector.root.name.name)

### Moving it

A robot's joints are connections, exactly like the drawer's slider — there is no separate
"robot motion" API. Grab one by name and set its `position`:

In [ ]:
shoulder = world.get_connection_by_name(PR2Joint.LEFT_SHOULDER_LIFT)
print("range:", shoulder.dof.limits.lower.position, "->", shoulder.dof.limits.upper.position)

shoulder.position = 1.0

ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Exercise 5 — move the other arm

You moved the left shoulder above. Now do the right one.

1. Get the `PR2RightArm` annotation from `world` — same pattern as `left_arm` above — and
   assign it to `right_arm`. Print its end effector's root body name; it should look just
   like the left gripper's.
2. Get the connection for `PR2Joint.RIGHT_SHOULDER_LIFT` and assign it to `right_shoulder`.
3. Set `right_shoulder.position` to something inside its limits, and different from
   `shoulder.position` (the left one you already moved).
4. Render.

In [ ]:
right_arm = world.get_semantic_annotations_by_type(PR2RightArm)[0]
print("right arm's gripper:", right_arm.end_effector.root.name.name)

right_shoulder = world.get_connection_by_name(PR2Joint.RIGHT_SHOULDER_LIFT)
right_shoulder.position = -0.2

ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

In [ ]:
# Run this to check your answer.
if right_arm is ... or not isinstance(right_arm, PR2RightArm):
    raise ExerciseVerificationFailed("right_arm should be the PR2RightArm annotation.")

expected_connection = world.get_connection_by_name(PR2Joint.RIGHT_SHOULDER_LIFT)
if right_shoulder is not expected_connection:
    raise ExerciseVerificationFailed(
        "right_shoulder should be world.get_connection_by_name(PR2Joint.RIGHT_SHOULDER_LIFT)."
    )

lower = right_shoulder.dof.limits.lower.position
upper = right_shoulder.dof.limits.upper.position
if not (lower <= right_shoulder.position <= upper):
    raise ExerciseVerificationFailed(f"right_shoulder.position must be within {lower} to {upper}.")
if right_shoulder.position == 0:
    raise ExerciseVerificationFailed("Move the right shoulder — set right_shoulder.position to something nonzero.")
if right_shoulder.position == shoulder.position:
    raise ExerciseVerificationFailed("Move the right shoulder by a different amount than the left one.")

print("Correct.")
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Exercise 6 — a different robot

The pattern above works for any robot with a semantic annotation class, not just PR2. `HSRB`
(Toyota's Human Support Robot) has one arm instead of two, spawns in a fraction of a second,
and behaves exactly the same way.

1. Spawn an `HSRB` into `world`, placed clear of the PR2 you already added — try
   `x=12.0, y=2.5`. Assign the result to `hsrb`.
2. Get its `HSRBArm` annotation and assign it to `arm`.
3. Get the connection for `HSRBJoint.ARM_FLEX`, assign it to `arm_flex`, move it to something
   inside its limits, and render.

In [ ]:
hsrb = RobotSpecification(
    semantic_annotation_type=HSRB,
    world_T_odom=HomogeneousTransformationMatrix.from_xyz_rpy(x=12.0, y=2.5),
).spawn(world)

arm = world.get_semantic_annotations_by_type(HSRBArm)[0]

arm_flex = world.get_connection_by_name(HSRBJoint.ARM_FLEX)
arm_flex.position = -1.5

ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

In [ ]:
# Run this to check your answer.
if hsrb is ... or not isinstance(hsrb, HSRB):
    raise ExerciseVerificationFailed("hsrb should be an HSRB annotation.")
if arm is ... or not isinstance(arm, HSRBArm):
    raise ExerciseVerificationFailed("arm should be the HSRBArm annotation.")

expected_connection = world.get_connection_by_name(HSRBJoint.ARM_FLEX)
if arm_flex is not expected_connection:
    raise ExerciseVerificationFailed(
        "arm_flex should be world.get_connection_by_name(HSRBJoint.ARM_FLEX)."
    )

lower = arm_flex.dof.limits.lower.position
upper = arm_flex.dof.limits.upper.position
if not (lower <= arm_flex.position <= upper):
    raise ExerciseVerificationFailed(f"arm_flex.position must be within {lower} to {upper}.")
if arm_flex.position == 0:
    raise ExerciseVerificationFailed("Move arm_flex — set its position to something nonzero.")

print("Correct.")
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Exercise 7 — build your own robot

Every robot so far came from a URDF someone else wrote. You can also build one from nothing,
the same way you built the dresser in §2 — bodies and connections, no file.

`AbstractRobotPart` (the base of `Arm`, `Torso`, and the rest) refuses to be built that way on
purpose — its parts are meant to come from parsing a real robot's URDF. `MinimalRobot` is the
escape hatch: it takes a kinematic branch you already built and wraps it, with no requirement
that the parts be typed.

1. Create a new `World`. Inside `my_world.modify_world()`, spawn a box body for the base —
   `BodySpecification.box(name, scale).spawn(my_world)` — and a second box for an "arm",
   attached to the base by passing `connection_specification=RevoluteConnectionSpecification(
   axis=...)` to its own `.spawn(my_world, parent=..., parent_T_self=...)`.
2. Wrap the base with `MinimalRobot.from_branch_in_world(my_base)`.
3. Move the arm's joint — it hangs off `my_arm.parent_connection` — and render.

Assign your objects to `my_world`, `my_robot`, `my_base`, `my_arm`.

In [ ]:
my_world = World.create_with_root_body()

with my_world.modify_world():
    my_base = BodySpecification.box("my_base", Scale(0.2, 0.2, 0.1)).spawn(my_world)
    my_arm = BodySpecification.box(
        "my_arm", Scale(0.3, 0.05, 0.05),
        connection_specification=RevoluteConnectionSpecification(axis=Vector3.Z()),
    ).spawn(
        my_world,
        parent=my_base,
        parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=0.15, z=0.05),
    )

my_robot = MinimalRobot.from_branch_in_world(my_base)

my_arm.parent_connection.position = 1.0

ray_tracer = RayTracer(my_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

In [ ]:
# Run this to check your answer.
if my_robot is ... or not isinstance(my_robot, MinimalRobot):
    raise ExerciseVerificationFailed("my_robot should be a MinimalRobot.")
if my_base is ... or my_robot.root is not my_base:
    raise ExerciseVerificationFailed(
        "my_robot should be built from my_base with MinimalRobot.from_branch_in_world."
    )
if my_arm is ... or my_arm not in my_robot.bodies_of_branch:
    raise ExerciseVerificationFailed("my_arm should be part of my_robot's kinematic branch.")

joint = my_arm.parent_connection
if not joint.has_hardware_interface:
    raise ExerciseVerificationFailed(
        "The connection to my_arm isn't controllable — did you attach it with a "
        "RevoluteConnectionSpecification (or similar ActiveConnection1DOF specification), "
        "and build my_robot with MinimalRobot.from_branch_in_world afterwards?"
    )
if joint.position == 0:
    raise ExerciseVerificationFailed("Move the arm — set its connection's position to something nonzero.")

print("Correct.")
ray_tracer = RayTracer(my_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

That is the whole loop from this tutorial applied to robots instead of furniture: parse a
URDF (or build one by hand), get typed — or minimal — annotations for its parts, move it by
writing to a connection's `position`. Real motion — reaching for a handle, planning a
collision-free path — builds on exactly this, one layer up.

## 6. Spatial predicates

EQL's queries so far asked *what type* something is. The library also ships spatial
predicates in `semantic_digital_twin.reasoning.predicates` — plain Python callables that ask
*where* something is relative to something else: `Above`, `Below`, `LeftOf`, `InsideOf`,
`is_supported_by`, `visible`, `reachable`. They work standalone, the same way they work inside
`.where(...)`.

This is also where §4 and §5 meet. The apartment and the PR2 have shared a world since §5, but
nothing has actually asked them a question about each other yet.

### Is it resting on something?

`is_supported_by(supported, supporting)` checks contact and a bit of geometry — no physics
simulation. A tiny synthetic stack shows it cleanly, the same way §2 built a drawer in
isolation before touching the apartment:

In [ ]:
blocks_world = World.create_with_root_body()

with blocks_world.modify_world():
    lower = BodySpecification.box("lower", Scale(0.3, 0.3, 0.1)).spawn(blocks_world)
    upper = BodySpecification.box("upper", Scale(0.2, 0.2, 0.1)).spawn(
        blocks_world,
        parent=lower,
        parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(z=0.1),
    )
    floating = BodySpecification.box("floating", Scale(0.2, 0.2, 0.1)).spawn(
        blocks_world,
        parent=lower,
        parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(z=0.5),
    )

print("upper    is supported by lower:", is_supported_by(upper, lower))
print("floating is supported by lower:", is_supported_by(floating, lower))

### Can the robot reach that?

`reachable(pose, root, tip)` runs inverse kinematics from `root` to `tip` and reports whether
it converges — no motion, no hardware, just: does a solution exist. `pose` can be expressed
relative to any frame; here, relative to the gripper's own current pose, which makes "near"
and "far" easy to read.

*(IK here is local, not global — it refines from wherever the arm currently is, so the same
target can be reachable from one starting configuration and not another. §5 left the left arm
at an extreme shoulder angle, so the cell below resets it to neutral first.)*

In [ ]:
if "world" not in globals():
    world = URDFParser.from_file(str(APARTMENT)).parse()
if not world.semantic_annotations:
    WorldReasoner(world).reason()
if "pr2" not in globals():
    pr2 = RobotSpecification(
        semantic_annotation_type=PR2,
        world_T_odom=HomogeneousTransformationMatrix.from_xyz_rpy(x=9.0, y=2.5),
    ).spawn(world)

for connection in pr2.left_arm.active_connections:
    connection.position = 0.0

tool_frame = pr2.left_arm.end_effector.tool_frame

near = HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.2, y=0.1, reference_frame=tool_frame)
far = HomogeneousTransformationMatrix.from_xyz_rpy(x=10.0, y=10.0, reference_frame=tool_frame)

print("near offset reachable:", reachable(near, pr2.left_arm.root, tool_frame))
print("far  offset reachable:", reachable(far, pr2.left_arm.root, tool_frame))

*(`visible` and `InsideOf` work the same way, but getting a camera to actually face
something requires getting its orientation right, which is its own small kinematics problem —
outside what this tutorial has room for. `reachable` sidesteps that: only position matters.)*

### Exercise 8 — reachable or not

Pick two targets relative to `tool_frame`: one you expect the arm can reach, one you expect it
cannot. Assign them to `close_target` and `far_target`, and the results of `reachable(...)` to
`close_reachable` and `far_reachable`.

In [ ]:
close_target = HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.2, y=0.1, reference_frame=tool_frame)
far_target = HomogeneousTransformationMatrix.from_xyz_rpy(x=10.0, y=10.0, reference_frame=tool_frame)

close_reachable = reachable(close_target, pr2.left_arm.root, tool_frame)
far_reachable = reachable(far_target, pr2.left_arm.root, tool_frame)

print("close_reachable:", close_reachable)
print("far_reachable  :", far_reachable)

In [ ]:
# Run this to check your answer.
if not isinstance(close_target, HomogeneousTransformationMatrix):
    raise ExerciseVerificationFailed("close_target should be a HomogeneousTransformationMatrix.")
if not isinstance(far_target, HomogeneousTransformationMatrix):
    raise ExerciseVerificationFailed("far_target should be a HomogeneousTransformationMatrix.")
if close_reachable is not True:
    raise ExerciseVerificationFailed(
        "close_target should be reachable — try a smaller offset from tool_frame."
    )
if far_reachable is not False:
    raise ExerciseVerificationFailed(
        "far_target should NOT be reachable — try a bigger offset from tool_frame."
    )

print("Correct.")

## 7. Teaching the reasoner

Back in §1, 25 bodies slide on a prismatic joint. The reasoner reported 19 drawers. Six
sliding bodies are therefore not drawers, according to the rules. Let's see them.

In [ ]:
sliding_but_not_drawers = sliding - {d.root.name.name for d in drawers}
print(sorted(sliding_but_not_drawers))

Two of these we already know about — the `_out_fancy` doors from §1, correctly excluded.

The other four are named like drawers. Let's look at what is attached to each of them, and
compare with a drawer that *was* found:

In [ ]:
def children_of(body_name):
    body = world.get_body_by_name(body_name)
    children = body.child_kinematic_structure_entities
    if not children:
        return "   (nothing attached)"
    return "\n".join(
        f"   {c.name.name}  via {type(c.parent_connection).__name__}" for c in children
    )

for name in ["cabinet2_drawer_big", "coffee_table_drawer",
             "cabinet2_door_out_fancy", "cabinet6_drawer_top"]:
    print(f"{name}:")
    print(children_of(name))
    print()

There it is.

- `cabinet2_door_out_fancy` has a door attached on a **revolute** joint. It slides out, then
  swings open. Not a drawer — the reasoner is right.
- `cabinet6_drawer_top`, which *was* found, has a handle fixed to it.
- `cabinet2_drawer_big` and `coffee_table_drawer` have **nothing attached at all**. Whoever
  modelled this apartment did not give them a handle.

And that is the gap. Recall the definition from §2:

```python
class Drawer(Furniture, HasCaseAsRootBody, HasHandle, HasMechanicalJoint):
```

The rule implements exactly that: a drawer *has a handle*. These four slide, they sit inside
cabinets, a robot can open them by pulling — but no handle was modelled, so the rule cannot
see them. This is not a naming problem and no amount of better string matching would help.
The definition is a little too strict for the data.

So the truth for Exercise 1 is **23**: the 19 with handles, plus these 4 without. Neither
heuristic's 25 was right, and neither was the reasoner's 19.

## 8. Where to go next

What we did: took a URDF that knew only geometry, gave it a vocabulary of typed concepts, had
a rule base infer those concepts automatically, queried them, and then found a case the rules
got wrong and taught them the missing one. The robot can now be told "open the drawer".

The thing worth taking away is the last step. The heuristics in §1 were wrong and gave you
nothing to work with. The reasoner was also incomplete — but it could tell you exactly what
rule produced each answer, which is what made the gap findable and fixable in twenty lines.

What we skipped, and where to find it — the library ships a full Jupyter Book under
`cognitive_robot_abstract_machine/semantic_digital_twin/doc/` (17 worked examples, concept
chapters, and self-assessment quizzes):

| Topic | Guide |
|---|---|
| Transforms and the `A_T_B` convention | `examples/using_transformations.md` |
| Declarative world building | `examples/building_worlds_with_specifications.md` |
| Regions and supporting surfaces | `examples/regions.md` |
| Saving annotated worlds to SQL | `examples/persistence_of_annotated_worlds.md` |
| Physics simulation (MuJoCo) | `examples/physics_simulators.md` |
| Adding a new robot | `examples/adding_robots.md` |
| Free-space decomposition and path planning | `examples/graph_of_convex_sets.md` |
| Loading RoboCasa / ProcTHOR / PartNet scenes | `doc/datasets.md` |

To convert any of them into a runnable notebook:

```bash
jupytext --to notebook cognitive_robot_abstract_machine/semantic_digital_twin/examples/regions.md
```

The predecessor to this tutorial, on writing the URDF itself, is
[EASE Fall School 2024 — Creating an Environment URDF](https://github.com/IntEL4CoRo/ease_fall_school_2024).